# 00 — Data Acquisition: Tomato Ripeness Datasets (Kaggle)

**Dự án:** Smart Greenhouse AI — cà chua
**Notebook này thiết kế để chạy trên Kaggle, không chạy local.**

## Trước khi chạy
1. Panel bên phải: **Settings → Internet → ON** (bắt buộc để tải dataset).
2. Accelerator để **None** — notebook này chỉ tải/giải nén dữ liệu, không train.
3. Chạy xong, bấm **Save Version → Save & Run All (Commit)** để output ở `/kaggle/working/ai/datasets/` được lưu thành Kaggle Dataset, dùng làm input cho notebook audit/train sau.

## Nguồn dữ liệu (nhánh độ chín quả — `tomato_ripeness_v1`)
Theo `TONG_HOP_DATASET_DO_CHIN_CA_CHUA_PUBLIC.docx` và `HUONG_DAN_XAY_DUNG_DATASET_SMART_GREENHOUSE_AI.md`:

| # | Dataset | Cách tải | Vai trò |
|---|---|---|---|
| 1 | Laboro Tomato | curl trực tiếp (S3 zip) | Nguồn chính |
| 2 | AgRobTomato | curl trực tiếp (Zenodo zip) | Bổ sung — ảnh robot trong nhà kính |
| 3 | TomatoPlantfactoryDataset | curl trực tiếp (Mendeley zip) | Bổ sung green/red, mật độ che khuất cao |
| 4 | OpenField-BD | curl trực tiếp (Zenodo rar) | Test ngoài miền — **license chưa rõ, không dùng để train tới khi xác nhận** |
| 5 | Tomatoes Dataset (Kaggle) | Attach qua "Add Input" trên Kaggle | Classifier phụ Old/Damaged (không có bounding box) |

Bộ Roboflow-hosted (IEEE 6-mức-chín, Morpheus) và bộ bệnh lá **không** nằm trong notebook này — tách riêng vì cần Roboflow API key, sẽ làm ở notebook kế tiếp.

## Việc notebook này làm
1. Tải và đóng băng dữ liệu gốc (giữ nguyên zip/rar, tính SHA-256).
2. Giải nén vào `raw/<dataset_id>/`.
3. Ghi `manifests/sources.csv`.
4. Kiểm kê sơ bộ (đếm file ảnh/label) từng nguồn → `manifests/dataset_inventory.csv`.

**Chưa** làm ở bước này: remap class, audit trực quan, chia split — thuộc notebook `01_ripeness_dataset_audit.ipynb`.

In [1]:
import hashlib
import shutil
import subprocess
import time
from pathlib import Path

import pandas as pd

BASE = Path("/kaggle/working/ai/datasets")
RAW = BASE / "raw"
MANIFESTS = BASE / "manifests"

for d in [RAW, MANIFESTS, RAW / "_archives"]:
    d.mkdir(parents=True, exist_ok=True)

# Ngày chạy notebook lần đầu — cập nhật nếu tải lại/refresh dataset
DOWNLOAD_DATE = "2026-08-05"

print("BASE:", BASE)

BASE: /kaggle/working/ai/datasets


In [2]:
def download_file(urls, dest_path, retries_per_url=2, timeout=120):
    """urls: 1 URL (str) hoặc list các URL ứng viên, thử lần lượt tới khi thành công."""
    if isinstance(urls, str):
        urls = [urls]
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    if dest_path.exists() and dest_path.stat().st_size > 0:
        print(f"[skip] {dest_path.name} đã tồn tại ({dest_path.stat().st_size / 1e6:.1f} MB)")
        return dest_path

    last_stderr = ""
    for url in urls:
        for attempt in range(1, retries_per_url + 1):
            print(f"[download] {url} -> {dest_path} (lần {attempt})")
            result = subprocess.run(
                ["curl", "-L", "--fail", "--retry", "2", "-o", str(dest_path), url],
                capture_output=True, text=True, timeout=timeout,
            )
            if result.returncode == 0 and dest_path.exists() and dest_path.stat().st_size > 0:
                print(f"[ok] {dest_path.name} ({dest_path.stat().st_size / 1e6:.1f} MB)")
                return dest_path
            last_stderr = result.stderr[-500:]
            print(f"[fail] returncode={result.returncode} stderr={last_stderr}")
            time.sleep(3)
        print(f"[next] Chuyển sang URL ứng viên kế tiếp (nếu có) sau khi '{url}' thất bại.")

    raise RuntimeError(f"Không tải được từ bất kỳ URL nào trong {urls}. Lỗi cuối: {last_stderr}")


def sha256_file(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def extract_archive(archive_path, dest_dir):
    archive_path = Path(archive_path)
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    suffix = archive_path.suffix.lower()
    if suffix == ".zip":
        shutil.unpack_archive(str(archive_path), str(dest_dir), format="zip")
    elif suffix == ".rar":
        subprocess.run(["unrar", "x", "-y", str(archive_path), str(dest_dir) + "/"], check=True)
    else:
        raise ValueError(f"Định dạng chưa hỗ trợ: {suffix}")
    print(f"[extract] {archive_path.name} -> {dest_dir}")
    return dest_dir

### Chẩn đoán kết nối mạng
Chạy cell dưới **trước** nếu bất kỳ bước tải nào báo `Could not resolve host`. Cell so sánh 3 mức: một host phổ biến (`google.com`), một host S3 gốc (`s3.amazonaws.com`), và host cụ thể của Laboro.

- Nếu **cả 3 dòng đều FAIL** → Internet đang **OFF** trong Settings (nguyên nhân phổ biến nhất). Bật Internet ON, Restart session, chạy lại từ đầu.
- Nếu chỉ dòng cuối (Laboro) FAIL còn 2 dòng đầu OK → đúng là chỉ riêng host này bị chặn/lỗi trên phía Kaggle, cần dùng URL path-style S3 thay thế (xem cell tải Laboro bên dưới).

In [3]:
def check_host(url, label):
    result = subprocess.run(
        ["curl", "-sS", "-I", "--max-time", "15", url],
        capture_output=True, text=True,
    )
    status = "OK" if result.returncode == 0 else f"FAIL (returncode={result.returncode})"
    print(f"[{label}] {status}")
    if result.returncode != 0:
        print(f"       stderr: {result.stderr.strip()[-300:]}")
    else:
        print(f"       {result.stdout.splitlines()[0] if result.stdout else ''}")

print("== Kiểm tra Internet trong notebook này ==")
check_host("https://www.google.com", "google.com (baseline)")
check_host("http://s3.amazonaws.com", "s3.amazonaws.com (baseline S3)")
check_host("http://assets.laboro.ai.s3.amazonaws.com/laborotomato/laboro_tomato.zip", "assets.laboro.ai.s3.amazonaws.com (host cụ thể)")

== Kiểm tra Internet trong notebook này ==
[google.com (baseline)] OK
       HTTP/2 200 
[s3.amazonaws.com (baseline S3)] OK
       HTTP/1.1 405 Method Not Allowed
[assets.laboro.ai.s3.amazonaws.com (host cụ thể)] OK
       HTTP/1.1 200 OK


### 1. Laboro Tomato
Nguồn chính — 804 ảnh, 9.777 bounding box, license CC BY-NC-SA 4.0 (phi thương mại, chia sẻ tương tự).

**Lưu ý:** dùng `http://` (không phải `https://`) — tên bucket S3 `assets.laboro.ai` chứa dấu chấm nên chứng chỉ TLS wildcard của AWS không khớp domain khi truy cập qua HTTPS virtual-hosted-style. Trang GitHub gốc của dataset cũng công bố link ở dạng `http://`.

In [4]:
laboro_zip = download_file(
    [
        "http://assets.laboro.ai.s3.amazonaws.com/laborotomato/laboro_tomato.zip",
        "http://s3.amazonaws.com/assets.laboro.ai/laborotomato/laboro_tomato.zip",  # path-style, fallback nếu subdomain không resolve
    ],
    RAW / "_archives" / "laboro_tomato.zip",
)
extract_archive(laboro_zip, RAW / "laboro_tomato")

[download] http://assets.laboro.ai.s3.amazonaws.com/laborotomato/laboro_tomato.zip -> /kaggle/working/ai/datasets/raw/_archives/laboro_tomato.zip (lần 1)
[ok] laboro_tomato.zip (1645.4 MB)
[extract] laboro_tomato.zip -> /kaggle/working/ai/datasets/raw/laboro_tomato


PosixPath('/kaggle/working/ai/datasets/raw/laboro_tomato')

### 2. AgRobTomato
449 ảnh 1280x720, Pascal VOC, ảnh chụp bởi robot di chuyển trong hàng nhà kính.

In [5]:
agrob_zip = download_file(
    "https://zenodo.org/records/5596799/files/Dataset-Greenhouse_Tomato_AgRob.zip?download=1",
    RAW / "_archives" / "agrob_tomato.zip",
)
extract_archive(agrob_zip, RAW / "agrob_tomato")

[download] https://zenodo.org/records/5596799/files/Dataset-Greenhouse_Tomato_AgRob.zip?download=1 -> /kaggle/working/ai/datasets/raw/_archives/agrob_tomato.zip (lần 1)
[ok] agrob_tomato.zip (137.0 MB)
[extract] agrob_tomato.zip -> /kaggle/working/ai/datasets/raw/agrob_tomato


PosixPath('/kaggle/working/ai/datasets/raw/agrob_tomato')

### 3. TomatoPlantfactoryDataset (Mendeley)
520 ảnh, 9.112 quả (green/red), YOLO + Pascal VOC, CC BY 4.0.

Link tải trực tiếp bên dưới được lấy từ Mendeley public API ngày 2026-08-05 (DOI `10.17632/8h3s6jkyff.3`). Nếu link trả về lỗi 404 (do phiên bản dataset thay đổi), vào https://data.mendeley.com/datasets/8h3s6jkyff/3 → bấm **Download All** để lấy link mới rồi thay vào cell dưới.

In [6]:
plantfactory_zip = download_file(
    "https://data.mendeley.com/public-files/datasets/8h3s6jkyff/files/beabf9c1-06dd-44e7-848f-a4352035e34c/file_downloaded",
    RAW / "_archives" / "tomato_plantfactory.zip",
    timeout=300,
)
extract_archive(plantfactory_zip, RAW / "tomato_plantfactory")

[download] https://data.mendeley.com/public-files/datasets/8h3s6jkyff/files/beabf9c1-06dd-44e7-848f-a4352035e34c/file_downloaded -> /kaggle/working/ai/datasets/raw/_archives/tomato_plantfactory.zip (lần 1)
[ok] tomato_plantfactory.zip (3402.9 MB)
[extract] tomato_plantfactory.zip -> /kaggle/working/ai/datasets/raw/tomato_plantfactory


PosixPath('/kaggle/working/ai/datasets/raw/tomato_plantfactory')

### 4. OpenField-BD Tomato Maturity
600 ảnh, ~2.4 GB, đồng ruộng thật (không phải nhà kính). **License chưa hiển thị rõ trên Zenodo** — theo tài liệu dự án, chỉ dùng làm tập test ngoài miền / đánh giá robustness, KHÔNG đưa vào train hay dataset phát hành cho tới khi xác nhận license.

File `.rar` cần công cụ `unrar` — cài trước khi giải nén.

In [7]:
subprocess.run(["apt-get", "-qq", "update"], check=False)
subprocess.run(["apt-get", "-qq", "install", "-y", "unrar"], check=False)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


CompletedProcess(args=['apt-get', '-qq', 'install', '-y', 'unrar'], returncode=0)

In [8]:
openfield_archive = download_file(
    "https://zenodo.org/records/20176021/files/OF-Tomato-BD.rar?download=1",
    RAW / "_archives" / "openfield_bd.rar",
    timeout=900,  # ~2.4 GB, cần thời gian tải lâu hơn
)
extract_archive(openfield_archive, RAW / "openfield_bd")

[download] https://zenodo.org/records/20176021/files/OF-Tomato-BD.rar?download=1 -> /kaggle/working/ai/datasets/raw/_archives/openfield_bd.rar (lần 1)
[ok] openfield_bd.rar (2391.2 MB)

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /kaggle/working/ai/datasets/raw/_archives/openfield_bd.rar

Creating    /kaggle/working/ai/datasets/raw/openfield_bd/tomato_m_f   OK
Extracting  /kaggle/working/ai/datasets/raw/openfield_bd/tomato_m_f/data.yaml       0%  OK 
Creating    /kaggle/working/ai/datasets/raw/openfield_bd/tomato_m_f/test  OK
Creating    /kaggle/working/ai/datasets/raw/openfield_bd/tomato_m_f/test/images  OK
Extracting  /kaggle/working/ai/datasets/raw/openfield_bd/tomato_m_f/test/images/IMG20260204142800.jpg       0%  OK 
Extracting  /kaggle/working/ai/datasets/raw/openfield_bd/tomato_m_f/test/images/IMG20260204143146.jpg       0%  OK 
Extracting  /kaggle/working/ai/datasets/raw/openfield_bd/tomato_m_f/test/image

PosixPath('/kaggle/working/ai/datasets/raw/openfield_bd')

### 5. Tomatoes Dataset (Kaggle) — attach thủ công
7.226 ảnh, classification (Unripe/Ripe/Old/Damaged), CC0. Vì notebook đang chạy trên Kaggle, cách nhanh nhất là **Add Input** dataset `enalis/tomatoes-dataset` từ panel bên phải thay vì tải qua API/kaggle.json.

Cell dưới copy dữ liệu từ `/kaggle/input/` vào `raw/` nếu đã attach; nếu chưa, in cảnh báo và không lỗi (để không chặn phần còn lại của notebook).

In [9]:
kaggle_input = Path("/kaggle/input/tomatoes-dataset")
if kaggle_input.exists():
    dest = RAW / "kaggle_tomato_states"
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(kaggle_input, dest)
    print(f"[ok] Copy {kaggle_input} -> {dest}")
else:
    print(
        "[warn] Chưa attach dataset 'enalis/tomatoes-dataset'.\n"
        "Vào panel phải -> Add Input -> tìm 'Tomatoes Dataset' (enalis) -> Add, "
        "rồi chạy lại cell này."
    )

[warn] Chưa attach dataset 'enalis/tomatoes-dataset'.
Vào panel phải -> Add Input -> tìm 'Tomatoes Dataset' (enalis) -> Add, rồi chạy lại cell này.


### Kiểm kê, checksum và `sources.csv`

In [10]:
SOURCES = [
    dict(
        dataset_id="laboro_tomato",
        name="Laboro Tomato",
        url="https://github.com/laboroai/LaboroTomato",
        download_url="http://assets.laboro.ai.s3.amazonaws.com/laborotomato/laboro_tomato.zip",
        license="CC BY-NC-SA 4.0",
        task="detection+segmentation",
        archive=RAW / "_archives" / "laboro_tomato.zip",
        extracted=RAW / "laboro_tomato",
    ),
    dict(
        dataset_id="agrob_tomato",
        name="AgRobTomato",
        url="https://zenodo.org/records/5596799",
        download_url="https://zenodo.org/records/5596799/files/Dataset-Greenhouse_Tomato_AgRob.zip?download=1",
        license="CAN_XAC_MINH_TREN_ZENODO",
        task="detection",
        archive=RAW / "_archives" / "agrob_tomato.zip",
        extracted=RAW / "agrob_tomato",
    ),
    dict(
        dataset_id="tomato_plantfactory",
        name="TomatoPlantfactoryDataset",
        url="https://data.mendeley.com/datasets/8h3s6jkyff/3",
        download_url="https://data.mendeley.com/public-files/datasets/8h3s6jkyff/files/beabf9c1-06dd-44e7-848f-a4352035e34c/file_downloaded",
        license="CC BY 4.0",
        task="detection",
        archive=RAW / "_archives" / "tomato_plantfactory.zip",
        extracted=RAW / "tomato_plantfactory",
    ),
    dict(
        dataset_id="openfield_bd",
        name="OpenField-BD Tomato Maturity",
        url="https://zenodo.org/records/20176021",
        download_url="https://zenodo.org/records/20176021/files/OF-Tomato-BD.rar?download=1",
        license="CHUA_RO - khong dung train truoc khi xac nhan",
        task="detection",
        archive=RAW / "_archives" / "openfield_bd.rar",
        extracted=RAW / "openfield_bd",
    ),
    dict(
        dataset_id="kaggle_tomato_states",
        name="Tomatoes Dataset (Kaggle)",
        url="https://www.kaggle.com/datasets/enalis/tomatoes-dataset",
        download_url="Attach qua Kaggle Add Input",
        license="CC0",
        task="classification",
        archive=None,
        extracted=RAW / "kaggle_tomato_states",
    ),
]

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
LABEL_EXTS = {".txt", ".xml", ".json"}

rows = []
inventory_rows = []
for src in SOURCES:
    extracted = Path(src["extracted"])
    archive = src["archive"]

    checksum = ""
    if archive is not None and Path(archive).exists():
        checksum = sha256_file(archive)

    exists = extracted.exists()
    n_images = n_labels = 0
    if exists:
        for p in extracted.rglob("*"):
            if p.suffix.lower() in IMAGE_EXTS:
                n_images += 1
            elif p.suffix.lower() in LABEL_EXTS:
                n_labels += 1

    rows.append({
        "dataset_id": src["dataset_id"],
        "name": src["name"],
        "url": src["url"],
        "download_url": src["download_url"],
        "version": "",
        "download_date": DOWNLOAD_DATE,
        "license": src["license"],
        "task": src["task"],
        "sha256": checksum,
        "extracted_path": str(extracted) if exists else "",
        "notes": "",
    })
    inventory_rows.append({
        "dataset_id": src["dataset_id"],
        "extracted": exists,
        "n_image_files": n_images,
        "n_label_like_files": n_labels,
    })

sources_df = pd.DataFrame(rows)
inventory_df = pd.DataFrame(inventory_rows)

sources_df.to_csv(MANIFESTS / "sources.csv", index=False)
inventory_df.to_csv(MANIFESTS / "dataset_inventory.csv", index=False)

print(sources_df[["dataset_id", "license", "sha256"]].to_string(index=False))
print()
print(inventory_df.to_string(index=False))

          dataset_id                                       license                                                           sha256
       laboro_tomato                               CC BY-NC-SA 4.0 0645b37864ac4610050f88b8e9163b5d8ed1c64159b357f0c93ba8e26e66b6a7
        agrob_tomato                      CAN_XAC_MINH_TREN_ZENODO c35d282373ac76ddc9889a8c300e5fc5b959462bb083fe77adcd7b0729c94766
 tomato_plantfactory                                     CC BY 4.0 e6bcaf1dfd6be32f62eadcfbfa99692e9feca6171cfcbfb5e54f728229905faf
        openfield_bd CHUA_RO - khong dung train truoc khi xac nhan 45f859670af746bbb061d63c25bb9bb57a85590d5e3fad0182f6965c40aa37c4
kaggle_tomato_states                                           CC0                                                                 

          dataset_id  extracted  n_image_files  n_label_like_files
       laboro_tomato       True            804                   2
        agrob_tomato       True            449                 451
 tomat

## Kết quả
- Dữ liệu gốc đã tải/giải nén vào `/kaggle/working/ai/datasets/raw/<dataset_id>/`.
- File zip/rar gốc giữ nguyên trong `raw/_archives/`, checksum SHA-256 ghi trong `manifests/sources.csv` — không sửa trực tiếp trong `raw/`.
- `manifests/dataset_inventory.csv` cho biết sơ bộ số file ảnh/label mỗi nguồn — dùng để phát hiện nguồn tải lỗi/thiếu trước khi audit chi tiết.

## Trước khi Save Version
- Kiểm tra `dataset_inventory.csv`: nếu `n_image_files = 0` ở nguồn nào đó, tải lại nguồn đó (thường do link hỏng hoặc giải nén sai định dạng).
- Nếu chưa attach `enalis/tomatoes-dataset`, thêm qua Add Input rồi chạy lại cell tương ứng trước khi Save Version.
- `openfield_bd` license chưa rõ — giữ nguyên trạng thái "không dùng train" cho tới khi có xác nhận license.

## Bước tiếp theo
Notebook `01_ripeness_dataset_audit.ipynb` (chưa tạo) sẽ:
1. Attach output của notebook này làm input.
2. Audit trực quan ≥100 ảnh/nguồn.
3. Chuẩn hóa `class_mapping.yaml` (gộp về `fruit_green_unripe` / `fruit_turning` / `fruit_ripe`).
4. Convert Pascal VOC → YOLO cho AgRobTomato và TomatoPlantfactory (nếu dùng VOC).
5. Phát hiện ảnh trùng (SHA-256 + pHash) và chia split chống leakage theo nguồn/video.